# 12_Task8_Step1 — Bước 1: xoay bàn tay về một hướng chuẩn (Task 8)

Notebook này **không sửa** 07_TrainWide và 08_WebcamWide. Nó gồm 3 phần: **train bản mới → so với bản mới nhất `wide45_lownoise` → chạy webcam để bạn tự kiểm chứng** (cell 8 → 11).

**Bước này làm gì (nói đơn giản):** trước đây model phải tự học "cùng một cử chỉ nhìn từ nhiều góc là một". Giờ mỗi khung hình, máy tự xoay bàn tay sao cho lòng bàn tay luôn quay về một hướng cố định (tính từ 3 điểm: cổ tay, gốc ngón trỏ, gốc ngón út). Góc nhìn gần như bị xóa, model chỉ còn phải nhìn hình dạng tay.

**Thay đổi so với `wide45_lownoise`:**
1. Thêm bước xoay chuẩn (tính trên GPU, rất nhẹ).
2. Bỏ augmentation xoay 3D vì đã dư (xoay xong cũng bị xoay lại). Giữ **lật gương** và **nhiễu 0.005**.
3. Sửa méo tỉ lệ khung hình (x chia cho chiều rộng, y chia cho chiều cao) trước khi xoay, vì video quay 1920x1080 còn webcam 640x480. Webcam tự lấy tỉ lệ từ khung hình.
4. Train tối đa **100 epoch**, dừng sớm **18 epoch**.

Checkpoint mới: `Data_321/checkpoints_task8/step1/TCN.pt` (không đè checkpoint cũ). Chạy lại sẽ tự nạp, không train lại.

**Cách chạy:** kích hoạt môi trường `C:\envs\train-gpu\Scripts\activate.bat`, mở notebook bằng kernel `train-gpu`, chạy lần lượt cell 1 → 11. Cell 5 là cell train (lâu nhất). Cell 7 in kết luận GIỮ hay BỎ. Cell 11 mở cửa sổ webcam.

**Phím trong cửa sổ webcam:** `q` thoát | `[` giảm ngưỡng | `]` tăng ngưỡng (mỗi lần 0.05) | `m` đổi qua lại giữa bản MOI (step1) và bản CU (wide45_lownoise) để so sánh ngay trên tay bạn.

In [1]:
#cell 1: đường dẫn, kiểm tra các file cần có, bật CUDA, cấu hình bước 1
import math, time, collections
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook này không chạy trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
TASK = BASE / "hand_landmarker.task"
CKPT_OLD = BASE / "checkpoints_v2" / "wide45_lownoise" / "TCN.pt"     # bản mới nhất hiện tại (để so sánh)
CKPT_NEW = BASE / "checkpoints_task8" / "step1" / "TCN.pt"            # bản bước 1
for p in (BASE / "manifest.csv", TASK, CKPT_OLD):
    assert p.exists(), f"Không thấy: {p}"
CKPT_NEW.parent.mkdir(parents=True, exist_ok=True)

ASPECT_TRAIN = 1080 / 1920      # cao / rộng của video quay (1920x1080); webcam tự lấy theo khung hình
CFG = {"palm_norm": True, "aspect": ASPECT_TRAIN, "noise": 0.005, "mirror": True, "epochs": 100, "patience": 18}
print("OK | GPU:", torch.cuda.get_device_name(0), "| checkpoint bước 1:", "đã có" if CKPT_NEW.exists() else "chưa có (cell 5 sẽ train)")

OK | GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU | checkpoint bước 1: chưa có (cell 5 sẽ train)


In [2]:
#cell 2: đọc manifest và dựng tensor train/val/test (giống 07: 126 số mỗi khung, cổ tay làm gốc, chia cỡ bàn tay)
m = pd.read_csv(BASE / "manifest.csv")

def feat(f):
    lm = np.load(BASE / f)["lm"]
    lm = lm - lm[:, :1]                                           # lấy cổ tay làm gốc
    scale = np.linalg.norm(lm[:, 9], axis=1)[:, None, None]
    lm = lm / scale                                               # chuẩn hóa theo cỡ bàn tay
    pos = lm.reshape(len(lm), 63)
    vel = np.vstack([np.zeros((1, 63)), np.diff(pos, axis=0)])
    has = ~np.isnan(pos[:, 0])
    return np.nan_to_num(np.hstack([pos, vel])).astype(np.float32), has

Tmax = int(m.khung.max())
def build(df):
    X = np.zeros((len(df), Tmax, 126), np.float32)
    M = np.zeros((len(df), Tmax), bool)
    for i, f in enumerate(df.path):
        x, h = feat(f)
        X[i, :len(x)] = x
        M[i, :len(x)] = h
    return (torch.tensor(X, device=dev),
            torch.tensor(df.nhan.values, device=dev),
            torch.tensor(M, device=dev))

meta = {t: m[m.tap == t].reset_index(drop=True) for t in ["train", "val", "test"]}
data = {t: build(meta[t]) for t in meta}
print("Tmax:", Tmax)
for t, (X, y, M) in data.items():
    print(t, tuple(X.shape), "| video:", len(y), "| khung có tay:", int(M.sum().item()))

Tmax: 355
train (151, 355, 126) | video: 151 | khung có tay: 19108
val (31, 355, 126) | video: 31 | khung có tay: 3782
test (33, 355, 126) | video: 33 | khung có tay: 3930


In [3]:
#cell 3: kiến trúc TCN (giống hệt 07 / 08, đầu vào vẫn 126 số)
class Block(nn.Module):
    def __init__(s, c, d):
        super().__init__(); s.pad = 2 * d; s.conv = nn.Conv1d(c, c, 3, dilation=d); s.drop = nn.Dropout(0.1)
    def forward(s, x):
        return x + s.drop(F.relu(s.conv(F.pad(x, (s.pad, 0)))))

class TCN(nn.Module):
    def __init__(s, c=64):
        super().__init__(); s.inp = nn.Linear(126, c)
        s.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)])
        s.out = nn.Linear(c, 3)
    def forward(s, x):
        return s.out(s.blocks(s.inp(x).transpose(1, 2)).transpose(1, 2))

CLASSES = ["paper", "rock", "scissors"]          # nhãn 0, 1, 2 theo thứ tự chữ cái
print("Đã định nghĩa TCN.")

Đã định nghĩa TCN.


In [4]:
#cell 4: BƯỚC 1 — xoay bàn tay về hướng chuẩn, hàm xoay 3D để kiểm tra, và tự kiểm tra hàm
pos_of = lambda X: X[..., :63].reshape(X.shape[0], X.shape[1], 21, 3)       # (b,T,126) -> (b,T,21,3)

def to_hand_units(pos, aspect):
    """pos (...,21,3) đã lấy cổ tay làm gốc: sửa méo tỉ lệ khung hình rồi chia lại cho cỡ bàn tay (mất tay = 0 thì vẫn là 0)."""
    p = pos * torch.tensor([1.0, aspect, 1.0], device=pos.device)
    return p / p[..., 9, :].norm(dim=-1)[..., None, None].clamp(min=1e-6)

def canon_feat(P, M):
    """P (b,T,21,3), M (b,T) có tay -> (b,T,126). Mỗi khung dựng hệ trục theo lòng bàn tay rồi đổi tọa độ sang hệ đó, thêm vận tốc."""
    b, T = P.shape[:2]
    n = F.normalize(torch.linalg.cross(P[:, :, 5], P[:, :, 17], dim=-1), dim=-1, eps=1e-6)   # trục z: vuông góc lòng bàn tay (cổ tay, gốc ngón trỏ, gốc ngón út)
    p9 = P[:, :, 9]
    u = F.normalize(p9 - (p9 * n).sum(-1, keepdim=True) * n, dim=-1, eps=1e-6)               # trục y: từ cổ tay lên gốc ngón giữa
    r = torch.linalg.cross(u, n, dim=-1)                                                      # trục x: ngang bàn tay
    R = torch.stack([r, u, n], dim=-2)                                                        # (b,T,3,3)
    pos = (P @ R.transpose(-1, -2)) * M[:, :, None, None]
    vel = torch.zeros_like(pos)
    vel[:, 1:] = (pos[:, 1:] - pos[:, :-1]) * (M[:, 1:] & M[:, :-1])[:, :, None, None]
    return torch.cat([pos.reshape(b, T, 63), vel.reshape(b, T, 63)], dim=-1)

def feats_new(X, M, aspect=ASPECT_TRAIN):
    return canon_feat(to_hand_units(pos_of(X), aspect), M)

def rot3d(pos, ax_deg, ay_deg, az_deg):
    """Xoay 3D (độ) quanh 3 trục, giống 06 / 07. Góc là số hoặc tensor (b,1,1)."""
    ax, ay, az = [torch.as_tensor(a, dtype=torch.float32, device=pos.device) * math.pi / 180 for a in (ax_deg, ay_deg, az_deg)]
    xx, yy, zz = pos[..., 0], pos[..., 1], pos[..., 2]
    y1, z1 = yy * ax.cos() - zz * ax.sin(), yy * ax.sin() + zz * ax.cos()
    x2, z2 = xx * ay.cos() + z1 * ay.sin(), -xx * ay.sin() + z1 * ay.cos()
    return torch.stack([x2 * az.cos() - y1 * az.sin(), x2 * az.sin() + y1 * az.cos(), z2], dim=-1)

def rotate_old(X, M, ax, ay, az):
    """Cách xoay đặc trưng cũ của 06 / 07 (dành cho bản cũ wide45_lownoise)."""
    b, T = X.shape[:2]
    pos = rot3d(pos_of(X), ax, ay, az) * M[:, :, None, None]
    vel = torch.zeros_like(pos)
    vel[:, 1:] = (pos[:, 1:] - pos[:, :-1]) * (M[:, 1:] & M[:, :-1])[:, :, None, None]
    return torch.cat([pos.reshape(b, T, 63), vel.reshape(b, T, 63)], dim=-1)

# Tự kiểm tra: xoay cứng bàn tay bao nhiêu độ thì đặc trưng sau chuẩn hóa vẫn phải giữ nguyên
Xv, yv, Mv = data["val"]
Pv = to_hand_units(pos_of(Xv), ASPECT_TRAIN)
ang = [torch.rand(len(Xv), 1, 1, device=dev) * 120 - 60 for _ in range(3)]
err = (canon_feat(rot3d(Pv, *ang) * Mv[:, :, None, None], Mv) - canon_feat(Pv, Mv)).abs().max().item()
print(f"Sai lệch tối đa sau khi xoay ngẫu nhiên ±60°: {err:.2e}")
assert err < 1e-3, "Chuẩn hóa hướng chưa bất biến theo góc xoay, kiểm tra lại canon_feat."

Sai lệch tối đa sau khi xoay ngẫu nhiên ±60°: 9.54e-07


In [5]:
#cell 5: train bản bước 1 (CUDA, tối đa 100 epoch, dừng sớm 18; augmentation chỉ còn lật gương + nhiễu; nạp lại nếu đã có)
def loss_acc(net, X, y, M):
    out = net(X)
    loss_each = F.cross_entropy(out.transpose(1, 2), y[:, None].expand(-1, X.shape[1]), reduction="none")
    correct = out.argmax(-1) == y[:, None]
    return (loss_each * M).sum() / M.sum().clamp(min=1), (correct & M).sum() / M.sum().clamp(min=1)

def aug_new(Xb, Mb, noise_std):
    """Lật gương (50%) + nhiễu tọa độ trên bàn tay đã sửa tỉ lệ, rồi mới xoay về hướng chuẩn và tính vận tốc."""
    b = Xb.shape[0]
    P = to_hand_units(pos_of(Xb), CFG["aspect"])
    flip = torch.rand(b, 1, 1, device=Xb.device) < 0.5
    P[..., 0] = torch.where(flip, -P[..., 0], P[..., 0])
    P = P + torch.randn_like(P) * noise_std * Mb[:, :, None, None]
    return canon_feat(P, Mb)

val_new = (feats_new(Xv, Mv), yv, Mv)       # val không augmentation, tính một lần

FORCE_RETRAIN = False   # True nếu muốn train lại bản bước 1

def train_new(bs=16, force=False):
    torch.manual_seed(0)
    net = TCN().to(dev)
    if CKPT_NEW.exists() and not force:
        ck = torch.load(CKPT_NEW, map_location=dev, weights_only=False)
        if ck.get("cfg") == CFG:
            net.load_state_dict(ck["w"])
            print(f"Nạp checkpoint có sẵn | epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f}")
            return net.eval()
        print("Checkpoint cũ có cấu hình khác, train lại.")

    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
    Xt, yt, Mt = data["train"]
    best, wait, hist, t0 = float("inf"), 0, [], time.time()
    for ep in range(1, CFG["epochs"] + 1):
        net.train(); perm = torch.randperm(len(Xt), device=dev); total = 0.0
        for i in range(0, len(Xt), bs):
            ids = perm[i:i + bs]
            xb = aug_new(Xt[ids], Mt[ids], CFG["noise"])
            loss, _ = loss_acc(net, xb, yt[ids], Mt[ids])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            total += loss.item() * len(ids)
        net.eval()
        with torch.no_grad(): vl, va = loss_acc(net, *val_new)
        row = {"ep": ep, "train_loss": total / len(Xt), "val_loss": vl.item(), "val_acc": va.item()}
        hist.append(row)
        if row["val_loss"] < best:
            best, wait = row["val_loss"], 0
            torch.save({"w": net.state_dict(), "epoch": ep, "val_loss": best, "hist": hist,
                        "model": "TCN", "variant": "step1_palm", "cfg": CFG}, CKPT_NEW)
        else:
            wait += 1
        if ep % 5 == 0 or wait >= CFG["patience"] or ep == CFG["epochs"]:
            print(f"step1 ep{ep:03d} | train {row['train_loss']:.4f} | val {row['val_loss']:.4f} | val_acc {row['val_acc']:.3f}")
        if wait >= CFG["patience"]:
            print(f"Dừng sớm: val loss không giảm sau {CFG['patience']} epoch."); break
    ck = torch.load(CKPT_NEW, map_location=dev, weights_only=False)
    net.load_state_dict(ck["w"])
    print(f"Tốt nhất epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f} | {time.time() - t0:.0f}s | đã lưu: {CKPT_NEW}")
    return net.eval()

net_new = train_new(force=FORCE_RETRAIN)

step1 ep005 | train 0.3529 | val 0.2609 | val_acc 0.905
step1 ep010 | train 0.3107 | val 0.2151 | val_acc 0.924
step1 ep015 | train 0.2704 | val 0.2414 | val_acc 0.899
step1 ep020 | train 0.2527 | val 0.2067 | val_acc 0.929
step1 ep025 | train 0.2481 | val 0.4433 | val_acc 0.848
step1 ep030 | train 0.2260 | val 0.1896 | val_acc 0.933
step1 ep035 | train 0.2375 | val 0.4934 | val_acc 0.839
step1 ep040 | train 0.2390 | val 0.2682 | val_acc 0.872
step1 ep045 | train 0.2281 | val 0.2051 | val_acc 0.917
step1 ep050 | train 0.1841 | val 0.2118 | val_acc 0.929
step1 ep051 | train 0.1758 | val 0.2141 | val_acc 0.918
Dừng sớm: val loss không giảm sau 18 epoch.
Tốt nhất epoch 33 | val_loss 0.1840 | 3s | đã lưu: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_task8\step1\TCN.pt


In [6]:
#cell 6: nạp bản cũ wide45_lownoise (bản mới nhất hiện tại) để so sánh
net_old = TCN().to(dev)
ck_old = torch.load(CKPT_OLD, map_location=dev, weights_only=False)
net_old.load_state_dict(ck_old["w"]); net_old.eval()
print(f"Đã nạp bản cũ {CKPT_OLD.parent.name} | epoch {ck_old['epoch']} | cấu hình: {ck_old.get('cfg')}")

Đã nạp bản cũ wide45_lownoise | epoch 49 | cấu hình: {'rot': (40, 45, 40), 'noise': 0.005}


In [7]:
#cell 7: so sánh bản mới (step1) với bản cũ trên tập test xoay ±0/15/30/45/60° (cùng góc xoay cho cả 2 model) rồi in kết luận GIỮ / BỎ
Xte, yte, Mte = data["test"]
THR = 0.7
Pte = to_hand_units(pos_of(Xte), ASPECT_TRAIN)
Lte = torch.tensor(meta["test"].khung.values, device=dev)
Hte = (torch.arange(Tmax, device=dev)[None, :] < Lte[:, None] / 2) & Mte      # nửa đầu clip (khung có tay)
OLD, NEW = "cũ: wide45_lownoise", "mới: step1"

def evaluate(net, Xin):
    with torch.no_grad():
        conf, pred = net(Xin).softmax(-1).max(-1)
    ok = pred == yte[:, None]
    sel = (conf >= THR) & Mte
    half = sel & Hte
    n = lambda k: max(k.sum().item(), 1)
    return {"acc_all": (ok & Mte).sum().item() / n(Mte) * 100,
            "acc_conf": (ok & sel).sum().item() / n(sel) * 100,
            "acc_conf_nua_dau": (ok & half).sum().item() / n(half) * 100,
            "coverage": sel.sum().item() / n(Mte) * 100}

MAGS, REPEAT = [0, 15, 30, 45, 60], 5
g_rand = torch.Generator(device=dev).manual_seed(0)
rows = []
for mag in MAGS:
    acc = {OLD: [], NEW: []}
    for _ in range(1 if mag == 0 else REPEAT):
        a = [torch.zeros(1, 1, 1, device=dev)] * 3 if mag == 0 else \
            [(torch.rand(len(Xte), 1, 1, device=dev, generator=g_rand) * 2 - 1) * mag for _ in range(3)]
        acc[OLD].append(evaluate(net_old, rotate_old(Xte, Mte, *a)))
        acc[NEW].append(evaluate(net_new, canon_feat(rot3d(Pte, *a) * Mte[:, :, None, None], Mte)))
    for name, lst in acc.items():
        rows.append({"model": name, "mag": mag, **{k: np.mean([r[k] for r in lst]) for k in lst[0]}})
res = pd.DataFrame(rows)

for metric, title in [("acc_all", "Độ chính xác mọi khung có tay (%)"),
                      ("acc_conf", f"Độ chính xác khi tự tin ≥ {THR} (%)"),
                      ("acc_conf_nua_dau", f"Độ chính xác khi tự tin ≥ {THR}, chỉ nửa đầu clip (%)"),
                      ("coverage", f"Độ phủ: % khung đạt ngưỡng {THR}")]:
    print(title, "— cột là ±mag (độ), 0 = không xoay")
    display(res.pivot_table(index="model", columns="mag", values=metric).reindex([OLD, NEW]).round(1))

p = res.pivot_table(index="model", columns="mag", values="acc_all")
d0, d45 = p.loc[NEW, 0] - p.loc[OLD, 0], p.loc[NEW, 45] - p.loc[OLD, 45]
ph = res.pivot_table(index="model", columns="mag", values="acc_conf_nua_dau")
dh = ph.loc[NEW, 0] - ph.loc[OLD, 0]
keep = (d0 >= -1) and (d45 > 0)
print(f"Δ acc 0°: {d0:+.1f} | Δ acc ±45°: {d45:+.1f} | Δ nửa đầu clip (tự tin ≥ {THR}, 0°): {dh:+.1f}")
print("KẾT LUẬN:", "GIỮ bản step1 (đạt: 0° không tụt quá 1 điểm và tốt hơn ở ±45°)" if keep
      else "BỎ bản step1, giữ wide45_lownoise (không đạt: 0° tụt quá 1 điểm hoặc không tốt hơn ở ±45°)")

Độ chính xác mọi khung có tay (%) — cột là ±mag (độ), 0 = không xoay


mag,0,15,30,45,60
model,,,,,
cũ: wide45_lownoise,92.5,92.3,91.3,91.2,90.4
mới: step1,91.4,91.4,91.4,91.4,91.4


Độ chính xác khi tự tin ≥ 0.7 (%) — cột là ±mag (độ), 0 = không xoay


mag,0,15,30,45,60
model,,,,,
cũ: wide45_lownoise,96.7,96.7,97.2,97.7,96.4
mới: step1,95.9,95.9,95.9,95.9,95.9


Độ chính xác khi tự tin ≥ 0.7, chỉ nửa đầu clip (%) — cột là ±mag (độ), 0 = không xoay


mag,0,15,30,45,60
model,,,,,
cũ: wide45_lownoise,94.3,94.3,95.3,96.0,93.6
mới: step1,92.6,92.6,92.6,92.6,92.6


Độ phủ: % khung đạt ngưỡng 0.7 — cột là ±mag (độ), 0 = không xoay


mag,0,15,30,45,60
model,,,,,
cũ: wide45_lownoise,89.4,89.2,87.2,85.9,84.6
mới: step1,90.5,90.5,90.5,90.5,90.5


Δ acc 0°: -1.1 | Δ acc ±45°: +0.2 | Δ nửa đầu clip (tự tin ≥ 0.7, 0°): -1.7
KẾT LUẬN: BỎ bản step1, giữ wide45_lownoise (không đạt: 0° tụt quá 1 điểm hoặc không tốt hơn ở ±45°)


In [8]:
#cell 8: MediaPipe HandLandmarker (CPU, chế độ VIDEO, num_hands=1) và hàm lấy tọa độ tay (giống 08)
import cv2
import mediapipe as mp

landmarker = mp.tasks.vision.HandLandmarker.create_from_options(mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(TASK)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO, num_hands=1))
T0 = time.perf_counter()     # mốc thời gian dùng chung: timestamp của MediaPipe phải luôn tăng, kể cả khi chạy lại cell 11

def detect(frame_bgr, ts_ms):
    """Trả về tọa độ ảnh chuẩn hóa (21,3) của tay (cùng loại tọa độ lúc train). Mất tay -> None."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    res_ = landmarker.detect_for_video(mp.Image(mp.ImageFormat.SRGB, rgb), ts_ms)
    if not res_.hand_landmarks:
        return None
    return np.array([[p_.x, p_.y, p_.z] for p_ in res_.hand_landmarks[0]], np.float32)
print("Đã tạo HandLandmarker (CPU).")

Đã tạo HandLandmarker (CPU).


In [9]:
#cell 9: nội suy webcam -> 120 fps, dựng đặc trưng cho CẢ 2 model (cũ như 08, mới có xoay chuẩn) và dự đoán khung cuối
FPS_MODEL, WIN = 120, 128     # TCN chỉ nhìn 63 khung gần nhất, nên cửa sổ 128 khung cho kết quả y như nhìn cả clip
buf = collections.deque(maxlen=WIN)          # mỗi phần tử (21,3); NaN = mất tay
state = {"prev_lm": None, "prev_t": None}

def push(lm, t):
    """Thêm một khung webcam (lm hoặc None nếu mất tay, t tính bằng giây) vào buf, chèn các khung nội suy 120 fps."""
    prev, pt = state["prev_lm"], state["prev_t"]
    n = 1 if pt is None else max(1, round((t - pt) * FPS_MODEL))
    if lm is None:
        buf.extend([np.full((21, 3), np.nan, np.float32)] * n)
    elif prev is None:
        buf.append(lm)
    else:
        buf.extend(prev + (lm - prev) * (k / n) for k in range(1, n + 1))
    state["prev_lm"], state["prev_t"] = lm, t

def hand_q(lm):
    """(T,21,3) -> cổ tay làm gốc, chia cỡ bàn tay (giống feat() ở cell 2). Mất tay = NaN."""
    lm = lm - lm[:, :1]
    return lm / np.linalg.norm(lm[:, 9], axis=1)[:, None, None]

def make_feat_old(lm):
    pos = hand_q(lm).reshape(len(lm), 63)
    vel = np.vstack([np.zeros((1, 63)), np.diff(pos, axis=0)])
    return np.nan_to_num(np.hstack([pos, vel])).astype(np.float32)

def make_feat_new(lm, aspect):
    M = torch.from_numpy(~np.isnan(lm[:, 0, 0]))[None].to(dev)
    Q = torch.from_numpy(np.nan_to_num(hand_q(lm)).astype(np.float32))[None].to(dev)
    return canon_feat(to_hand_units(Q, aspect), M)

def predict(which, aspect):
    """which = 'new' hoặc 'old'. Trả về (lớp, độ tin cậy) cho khung cuối, hoặc None nếu khung cuối không có tay."""
    lm = np.stack(buf)
    if np.isnan(lm[-1, 0, 0]):
        return None
    with torch.no_grad():
        if which == "new":
            p = net_new(make_feat_new(lm, aspect)).softmax(-1)[0, -1]
        else:
            p = net_old(torch.from_numpy(make_feat_old(lm))[None].to(dev)).softmax(-1)[0, -1]
    conf, c = p.max(0)
    return int(c), float(conf)

for w_ in ("new", "old"):       # làm nóng GPU để khung đầu không bị giật
    buf.clear(); buf.extend([np.zeros((21, 3), np.float32) + np.arange(21)[:, None].astype(np.float32) * 0.1] * 3)
    predict(w_, 0.75)
buf.clear()
print("Sẵn sàng: nội suy", FPS_MODEL, "fps | cửa sổ", WIN, "khung")

Sẵn sàng: nội suy 120 fps | cửa sổ 128 khung


In [10]:
#cell 10: giao diện giống 08 — màu theo lớp, chữ ĐEN trên nền màu, khung tay + khung xương, thêm dòng tên model đang chạy
COLORS = {0: (60, 60, 255), 1: (255, 150, 40), 2: (60, 210, 60)}   # BGR: paper = đỏ, rock = xanh dương, scissors = xanh lá
GRAY, YELLOW, BLACK = (190, 190, 190), (150, 255, 255), (0, 0, 0)
BONES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),
         (9,13),(13,14),(14,15),(15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]

def put_label(img, text, x, y, bg, scale=0.8):
    """Chữ đen trên hộp màu; (x, y) là góc dưới-trái của hộp."""
    (w, h), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    cv2.rectangle(img, (x, y - h - base - 6), (x + w + 10, y + 2), bg, -1)
    cv2.putText(img, text, (x + 5, y - base), cv2.FONT_HERSHEY_SIMPLEX, scale, BLACK, 2, cv2.LINE_AA)

def draw_hand(img, img_lm, color, label, mirror=True):
    """Vẽ khung xương, khung bao quanh tay và nhãn (cùng màu) lên ảnh."""
    h, w = img.shape[:2]
    px, py = (img_lm[:, 0] * w).astype(int), (img_lm[:, 1] * h).astype(int)
    if mirror: px = w - 1 - px
    for a, b in BONES:
        cv2.line(img, (px[a], py[a]), (px[b], py[b]), color, 3, cv2.LINE_AA)
    for i in range(21):
        cv2.circle(img, (px[i], py[i]), 6, color, -1, cv2.LINE_AA)
        cv2.circle(img, (px[i], py[i]), 6, BLACK, 1, cv2.LINE_AA)
    x0, y0, x1, y1 = px.min() - 15, py.min() - 15, px.max() + 15, py.max() + 15
    ly = y1 + 34 if (x0 < 330 and y0 < 165) else y0      # tránh đè lên khối FPS / ngưỡng / model ở góc trên trái
    put_label(img, label, max(x0, 0), min(max(ly, 32), h - 5), color)

def draw_hud(img, fps, thr, which):
    """FPS, ngưỡng, model đang chạy và chú thích màu của 3 lớp."""
    put_label(img, f"FPS {fps:.0f}", 10, 35, YELLOW)
    put_label(img, f"Nguong >= {thr:.2f}", 10, 75, YELLOW)
    put_label(img, "Model: MOI step1" if which == "new" else "Model: CU wide45", 10, 115, YELLOW)
    for i, name in enumerate(CLASSES):
        put_label(img, name, img.shape[1] - 125, img.shape[0] - 10 - 40 * (2 - i), COLORS[i])
print("Đã định nghĩa giao diện.")

Đã định nghĩa giao diện.


In [11]:
#cell 11: chạy webcam — q: thoát | [ ]: giảm/tăng ngưỡng | m: đổi qua lại giữa model MOI (step1) và model CU (wide45_lownoise)
CAM_ID, MIRROR, THR_CAM, WHICH = 0, True, 0.70, "new"

cap = cv2.VideoCapture(CAM_ID, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))     # MJPG giúp webcam đạt 30 fps ở 640x480
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640); cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480); cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
assert cap.isOpened(), "Không mở được webcam (thử đổi CAM_ID sang 1)."

buf.clear(); state.update(prev_lm=None, prev_t=None)
fps, last = 0.0, time.perf_counter()
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        now = time.perf_counter()
        aspect = frame.shape[0] / frame.shape[1]                    # cao / rộng của khung webcam (640x480 -> 0.75)
        img_lm = detect(frame, int((now - T0) * 1000))              # MediaPipe nhận ảnh gốc (không lật), giống lúc trích landmark
        push(img_lm, now)
        out = predict(WHICH, aspect) if img_lm is not None else None

        disp = cv2.flip(frame, 1) if MIRROR else frame              # chỉ lật để hiển thị cho dễ chơi
        fps, last = 0.9 * fps + 0.1 / max(now - last, 1e-6), now
        if out is None:
            put_label(disp, "no hand", 10, 155, GRAY)
        else:
            c, conf = out
            sure = conf >= THR_CAM
            draw_hand(disp, img_lm, COLORS[c] if sure else GRAY,
                      f"{CLASSES[c]} {conf:.0%}" if sure else f"{CLASSES[c]}? {conf:.0%}", MIRROR)
        draw_hud(disp, fps, THR_CAM, WHICH)

        cv2.imshow("RPS webcam - Task 8 step 1 (q: thoat, m: doi model)", disp)
        k = cv2.waitKey(1) & 0xFF
        if k == ord("q"): break
        if k == ord("["): THR_CAM = max(0.05, THR_CAM - 0.05)
        if k == ord("]"): THR_CAM = min(0.95, THR_CAM + 0.05)
        if k == ord("m"): WHICH = "old" if WHICH == "new" else "new"
finally:
    cap.release(); cv2.destroyAllWindows()
print("Đã đóng webcam.")

Đã đóng webcam.


## Cách đọc kết quả
- Cell 7 in kết luận **GIỮ** khi bản mới đạt cả hai tiêu chí giống 07: độ chính xác ở 0° không tụt quá 1 điểm so với `wide45_lownoise`, và tốt hơn ở ±45°. Nếu không thì **BỎ** và giữ `wide45_lownoise`.
- Tập test chỉ có 33 video, nên chênh dưới khoảng 2 điểm không đáng tin. Vì vậy hãy dùng cell 11: bấm `m` để đổi qua lại hai model ngay trên tay bạn, thử các góc lạ và tay trái / tay phải. Kết quả thật trên webcam mới là quyết định cuối cùng.
- Nếu bản mới không tốt hơn, nguyên nhân thường gặp là tọa độ z của MediaPipe còn nhiễu nên mặt phẳng lòng bàn tay ước lượng chưa ổn định. Khi đó ghi lại kết quả và chuyển sang Bước 2 trên nền `wide45_lownoise`.
- Chép 1–2 dòng kết quả (Δ 0°, Δ ±45°, kết luận GIỮ / BỎ, nhận xét webcam) vào phần Ghi chép tiến trình của Decision.docx.